<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 05 — Compare, and Report

**Paired with L12.2 · Prediction of Power Grid Stability**

Two surrogates for the same critical clearing time, one that was told which
line was out and one that was shown. This notebook puts the numbers side by
side, runs the one experiment none of the earlier notebooks could run alone,
and assembles the report.

What is marked is not whether your numbers match anyone else's. It is whether
you can say what you measured, what it means, and where it stops being true.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
import os
os.makedirs(pb.RESULTS, exist_ok=True)

data = pb.build_dataset(n_ops=180, seed=12)
cases = pb.contingencies()
cct, op_id, cont_id = data["cct"], data["op_id"], data["cont_id"]

nb02 = pb.load("nb02_dense")
nb03 = pb.load("nb03_graph")
nb04 = pb.load("nb04_screening")

train = nb02["train"].astype(bool)
test = nb02["test"].astype(bool)
y_true = cct[test]

preds = {"linear": nb02["lin_test"],
         "dense": nb02["pred_test"],
         "graph": nb03["pred_test"]}

rows = []
for name, p in preds.items():
    c = pb.confusion(y_true, p)
    rows.append([name,
                 f"{np.abs(y_true-p).mean()*1e3:.2f}",
                 f"{np.abs(y_true-p).max()*1e3:.1f}",
                 f"{c['missed']}", f"{c['alarms']}", f"{c['accuracy']:.3f}"])
print(pb.screening_table(rows))

print()
print(f"  permutation gap, graph : {float(nb03['gap_gnn'])*1e3:.3e} ms")
print(f"  permutation gap, dense : {float(nb03['gap_dense'])*1e3:.3f} ms")
print(f"  held-out topology MAE (linear, nb02): "
      f"{float(nb02['held_out_topology_mae'])*1e3:.2f} ms"
      f"   recall {float(nb02['held_out_topology_recall']):.3f}")

---

## 0b · Your personal seed

Every notebook in this set fixes the seed to 88 so the printed "what you should
see" blocks are true on any machine. That is right for checking your work and
wrong for reporting it — with one seed the whole cohort produces identical
numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints where the questions ask for it. Your supervisor can
regenerate exactly these numbers from your study number alone.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Three dispatches nobody else in the cohort will draw, screened properly.
your_ops = pb.operating_points(3, seed=SEED)

print()
print(f"  {'op':>3s}{'load':>9s}{'Gen east':>10s}"
      + "".join(f"{('c'+str(c['index'])):>8s}" for c in cases))
your_cct = np.zeros((3, pb.N_CONTINGENCY))
for i, o in enumerate(your_ops):
    for c in cases:
        your_cct[i, c["index"]] = pb.label_case(o, c["outage"], c["fault_bus"])
    print(f"  {i:>3d}{-o[0][[2,3,4]].sum():>9.3f}{o[0][1]:>10.3f}"
          + "".join(f"{v*1e3:>8.1f}" for v in your_cct[i]))

worst = your_cct.min(axis=1)
print()
print(f"  your worst clearing time      : {worst.min()*1e3:.1f} ms")
print(f"  your insecure cases           : "
      f"{int((your_cct <= pb.PROTECTION_TIME).sum())} of {your_cct.size}")
print(f"  your mean CCT                 : {your_cct.mean()*1e3:.2f} ms")
print(f"  (the shared dataset's mean is : {cct.mean()*1e3:.2f} ms)")

**What you should see.** Three dispatches, six clearing times each, all in
milliseconds, and a summary line. Times printed as `500.0` are censored at the
bisection ceiling and are not measurements — say so if you quote one.

---

## 1 · The question neither notebook asked

Notebook 02 held out a topology from the **linear** model and watched it fail.
Notebook 03 built a graph model and never subjected it to the same test. That
is the experiment this exercise set exists to run, and it is the table the
course notes have been promising since L12.2:

| | seen topologies | **held-out topology** |
|---|---|---|
| dense MLP | ? | ? |
| graph network | ? | ? |

The left column is the control and will look respectable for both. **The right
column is the result.**

A random split will not produce this table. You must hold out a whole topology
— train on contingencies where line 0 is never removed, test only on cases
where it is — or you are measuring interpolation and calling it generalisation.

### Your turn

In [ ]:
# TODO: retrain both architectures with the tie outage held out, and fill in
# the table.
#
#   seen = cont_id != 1          # every contingency except "trip line 0"
#   held = cont_id == 1
#
#   Standardise on `seen` ONLY -- both the flattened inputs for the dense model
#   and the per-channel node statistics for the graph model. Using the earlier
#   notebooks' statistics leaks the held-out topology into the preprocessing.
#
#   DENSE: exactly notebook 02's model and schedule, fitted on Z[seen].
#   GRAPH: exactly notebook 03's model and schedule, fitted on (A_hat, X)[seen].
#
#   For each, score on `seen` and on `held`:
#       mae   = np.abs(cct[mask] - prediction).mean()
#       bias  = (prediction - cct[mask]).mean()          # sign matters
#       conf  = pb.confusion(cct[mask], prediction)
#
#   Put them in
#       topo = {"dense": {"seen_mae":..., "held_mae":..., "held_bias":...,
#                         "held_missed":..., "held_recall":...,
#                         "held_pred":...},          # the predictions themselves
#               "graph": {...}}
#
#   "held_pred" must be the model's prediction on cct[cont_id == 1], in
#   seconds and in that order -- the next cell plots it.
#
#   Two training runs, a few minutes.
#
# Predict the sign of `held_bias` before you look. Line 0 is the only branch
# whose loss lengthens the electrical distance between the two machines, so a
# model that has never seen it removed is predicting for a stronger network
# than the one it is being asked about.

raise NotImplementedError("Run the held-out-topology experiment for both models")

In [ ]:
rows = [[name,
         f"{t['seen_mae']*1e3:.2f}",
         f"{t['held_mae']*1e3:.2f}",
         f"{t['held_mae']/t['seen_mae']:.1f}x",
         f"{t['held_bias']*1e3:+.1f}",
         f"{t['held_missed']}",
         f"{t['held_recall']:.3f}"]
        for name, t in topo.items()]
print(error_table(rows, ["model", "seen MAE [ms]", "held-out MAE [ms]",
                         "ratio", "held-out bias [ms]", "missed", "recall"]))

print()
print("  reference, linear baseline (notebook 02 section 5):")
print(f"    seen 24.03 ms   held-out {float(nb02['held_out_topology_mae'])*1e3:.2f} ms"
      f"   recall {float(nb02['held_out_topology_recall']):.3f}")
print(f"    truly insecure held-out cases: "
      f"{int((cct[cont_id == 1] <= pb.PROTECTION_TIME).sum())}")

fig, ax = plt.subplots(1, 2, figsize=(9.6, 4.4))
for a, (name, _t) in zip(ax, topo.items()):
    pb.plot_parity(cct[cont_id == 1], topo[name]["held_pred"], ax=a,
                   title=f"{name}, held-out topology")
fig.tight_layout(); plt.show()

**What you should see.** Both models worse on the held-out topology than on the
seen ones — that much is certain, and a result that showed otherwise would mean
the split is not doing what you think.

What is *not* certain, and is the finding of this exercise, is the ratio. The
argument of L12.2 predicts that the graph model degrades less, because the
change it is being asked to generalise across is expressed in its inputs: with
line 0 removed, buses 0 and 1 simply have one neighbour fewer, and it has seen
buses with one neighbour fewer in every other contingency. The dense model's
only channel for the same information is a bit that was zero in every training
row.

**If your graph model does not beat the dense one here, report it.** Then check,
in this order: depth — with the tie out the diameter is 4, so a 3-layer model is
one hop short on exactly the held-out topology; the standardisation, which must
come from `seen` alone; and the training-set size, which is 900 rather than 864
and should not be the problem. An honest negative, investigated, is worth more
than a tuned positive, and there is a real possibility here — six contingencies
is a very small number of topologies to learn structure from, and saying so is
part of the answer.

The reference number in the printout is the **linear** model on the same split:
**67.42 ms held-out against 24.03 ms seen, recall 0.264, 53 of 72 insecure
cases missed**. Anything you report should be read against that.

---

## 2 · Your answers

Replace every string. Keep to the word limits; they are tight on purpose.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_WHY_A_SURROGATE = """
(120 words) Notebook 00 timed one critical clearing time and notebook 01 timed
1080 of them. Quote both numbers. Then state precisely what the surrogate in
this exercise is allowed to replace and what it is not, and say what would have
to be true before you would let it replace more.
"""

Q2_THE_GRAPH_IS_AN_INPUT = """
(150 words) Explain why a line outage is a different INPUT to a graph network
and only a different LABEL to a dense one. Use the degree vectors from notebook
03 section 1 to make it concrete. Then say what the fault location is, in the
same vocabulary, and why it lives in a different channel from the outage.
"""

Q3_PERMUTATION = """
(120 words) Report both permutation gaps from notebook 03 with their units.
Explain why the graph model's is not exactly zero, and why that non-zero value
is nevertheless evidence of an exact property. Then name one situation in a real
energy management system where the bus ordering changes.
"""

Q4_HELD_OUT_TOPOLOGY = """
(150 words) Report your four numbers from section 1 and the sign of each
held-out bias. Say which architecture degraded less and by how much. If the
graph model did not win, say so and give the mechanism you think is responsible
-- that answer scores as well as the other one if it is specific.
"""

Q5_THE_ASYMMETRY = """
(150 words) Give your zero-miss screening threshold and what it cost in false
alarms. Then argue the choice to somebody who wants the threshold lowered
because the false-alarm count is embarrassing. Quote at least one number from
notebook 04 section 2 and one from your personal-seed dispatches in section 0b.
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result in this exercise you trust least, and say exactly
what experiment would settle it. Candidates worth considering: the constant-
impedance load model, the censored labels at 500 ms, the classical machine
model with two machines, and the fact that the line impedances are plausible
rather than measured. An answer naming a specific number and a specific test
scores higher than a general statement about needing more data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

In [ ]:
answers = {
    "1 · Why a surrogate at all": (Q1_WHY_A_SURROGATE, 120),
    "2 · The graph is an input": (Q2_THE_GRAPH_IS_AN_INPUT, 150),
    "3 · The permutation test": (Q3_PERMUTATION, 120),
    "4 · Held-out topology": (Q4_HELD_OUT_TOPOLOGY, 150),
    "5 · The asymmetry that sets the threshold": (Q5_THE_ASYMMETRY, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_12.2 — Prediction of Power Grid Stability", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "> The six-bus case is DK2-REPRESENTATIVE: the structure follows",
             "> eastern Denmark, the line impedances are plausible values rather",
             "> than measured ones. State which of your conclusions depend on them.",
             "", "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)
    out = os.path.join(pb.RESULTS, "Ex12.2_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.2_report.md into Ex12.2_report.pdf, with any figure
# saved as Ex12.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex12.2_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex12.2_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex12.2_report.pdf")
print("written Ex12.2_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex12.2_report.pdf")
except ImportError:
    pass


## 4 · What Ex_12.2 was for

One question — *which outage would we not survive?* — asked of one network,
answered twice, with the difference between the two answers being **where the
topology was put**.

* **A one-hot flag names a topology.** The model can memorise which name is
  dangerous. It cannot generalise to a name it has not seen, and it cannot
  survive somebody renumbering the buses.
* **An adjacency describes one.** A line out is one fewer neighbour, in the same
  variables as every other neighbourhood, so a change of topology is a change of
  input rather than a change of problem.
* **Permutation invariance is architectural, not learned.** It holds before
  training, it holds exactly, and nothing in the loss asked for it. That is what
  distinguishes a guarantee from a tendency.
* **Depth is the graph's diameter.** Here 3, and 4 in the contingency that
  matters. Not a hyperparameter — a measurement.
* **The two errors of a screen are not commensurable.** A missed insecure case
  is a blackout; a false alarm is an afternoon. Any metric that averages them
  has thrown away the only distinction that matters.

And the one that outlives the exercise: **a surrogate replaces a sweep, not a
simulator**. Everything the screen flags still gets simulated properly. The
value of the model is that it decides what to spend the simulator on — which is
a decision, not a prediction, and is scored accordingly.

This closes Part 2. Ex_07 put a known condition into the function space rather
than the loss; Ex_12.2 put a known structure into the architecture rather than
the data. They are the same move, made twice, five weeks apart.